# Models, Context & Agents

An AI application is easier to understand when we separate **what generates**, **what supplies evidence**, and **what controls actions**. A language model may sit inside a retrieval system, a fixed workflow, or an agent. Those surrounding choices change what the application can accomplish.

```{figure} figures/ai-layers.svg
:alt: Sources provide selected evidence to context; a model generates responses inside an agent loop.

Three responsibilities: evidence selection, generation, and control. An agent can repeat the process as new tool results arrive.
```


## The Model Layer

Artificial intelligence
: The broad field of building systems that perform tasks associated with intelligence.

| Term | What the label describes | Example question |
|---|---|---|
| Machine learning | Learning patterns from data | Can this system learn a predictor? |
| Deep learning | Methods based on multilayer neural networks | What model family is used? |
| Generative AI | Producing content | Can it generate text, images, or audio? |
| Large language model (LLM) | A model centered on language processing and generation | Can it interpret and produce language? |
| Multimodal system | Working across multiple types of input or output | Can it relate an image to a textual question? |

:::{admonition} A useful progression, not a strict family tree
:class: warning

AI → machine learning → deep learning → generative AI is a useful learning route, but the labels do not all describe the same property. “Generative” describes an output capability; “deep learning” describes a family of methods. An interface that accepts an image also does not prove that one jointly trained model handles every modality.
:::

Choose capabilities according to the task. Breadth alone does not determine quality: latency, cost, privacy, context capacity, and available tools also matter. A text-only analysis task may not benefit from a model that can generate several media types.


## Context Is a Working Space

Context window
: The finite token capacity available for the model's current instructions, inputs, conversation, evidence, and output. Tokens are pieces of encoded content, not a fixed number of words.

A simplified budget is

$$C_{\mathrm{used}}=C_{\mathrm{instructions}}+C_{\mathrm{history}}+C_{\mathrm{evidence}}+C_{\mathrm{output}}\leq C_{\mathrm{window}}.$$

The output term is an allowance that must fit the request's limits; the exact accounting depends on the system. The application decides which old messages and sources are carried forward.

:::{admonition} A context budget in numbers
:class: tip ds-example-card

For an **illustrative** 16,000-token budget, reserve 2,000 tokens for instructions, 4,000 for conversation history, and 2,000 for output. That leaves 8,000 for evidence. A 30,000-token log cannot fit unchanged: retrieve relevant sections, summarize carefully, or analyze parts separately.

This is budgeting arithmetic, not a specification for any model.
:::

Context is closer to a workbench than permanent memory. Putting a document on the workbench makes it available now; it does not change the trained parameters. A product may separately store information and retrieve it into a future request.

:::{admonition} Compression can lose the deciding detail
:class: note

A summary can drop an exception, a timestamp, or a source identifier. Some models also use information at different positions in a long prompt unevenly. More context is not automatically better evidence: preserve the facts needed for the decision and test whether they are actually used.
:::


## In-Context Learning and RAG

:::{div}
:class: definition-entry

In-context learning
: Using supplied instructions, examples, or task material during inference without updating the model's parameters. Demonstrations of a task are a common form; the supplied paper uses the term broadly for temporary task knowledge.

:::

:::{div}
:class: definition-entry

Retrieval augmented generation (RAG)
: Retrieving evidence and putting it into the context of a generative model. Vector search is one implementation; keyword search and other retrieval methods can also supply the evidence.

:::

```{figure} figures/rag-pipeline.svg
:alt: Chunk source material, embed and index it, retrieve relevant chunks, then generate a source-supported answer.

Retrieval decides which evidence gets space in the context. It does not permanently teach the generator the collection.
```

1. **Chunk:** divide documents into coherent units while keeping important qualifiers.
2. **Embed and index:** store numerical representations together with text identifiers and metadata.
3. **Search:** encode the question in a compatible space and rank candidate passages. Filters or keyword search can help.
4. **Assemble context:** include the question, selected text, and source references.
5. **Generate and check:** ask for an evidence-supported answer and verify its claims.

For nonzero query and passage vectors $q$ and $d$, one similarity measure is

$$\operatorname{sim}(q,d)=\frac{q\cdot d}{\lVert q\rVert\lVert d\rVert}.$$

The dot product measures alignment; normalization removes dependence on vector length. High similarity indicates a learned relationship, not that a passage is true or sufficient to answer the question.

:::{admonition} Where retrieval can fail
:class: warning

A chunk can lose the condition attached to a claim. An acronym can be unfamiliar to the encoder. A version filter can exclude the needed source. Several near-duplicate results can crowd out a decisive passage. Inspect retrieved text before blaming the generator.
:::

Very small chunks can be precise but incomplete. Very large chunks preserve surrounding explanation but use more context and can match for the wrong reason. Chunk boundaries, overlap, ranking, and question types must be considered together.


## From a Response to an Agent

Agent
: A goal-directed system that chooses steps, uses available tools, inspects results, and decides whether to continue or stop.

```{figure} figures/agent-cycle.svg
:alt: Plan, act through a tool, observe the result, and decide whether to stop or plan again.

The feedback loop distinguishes an investigation from a single generated response.
```

A model call can occur at every step. Agency therefore depends on the surrounding control loop as well as the model. Tool access allows the system to inspect or change external state; the generated text alone does not perform that action.

::::{div}
:class: reading-comparison

:::{div}
### Fixed Workflow

Most transitions are specified in advance. Useful when the process is known and repeatability is important: fetch a report, validate its schema, calculate totals, and format the result.
:::

:::{div}
### Adaptive Agent

The next step depends on discovered evidence. Useful when a failed build may require reading a log, a configuration file, or a recent change before deciding what to inspect next.
:::
::::

A production system can combine both: fixed validation and approval steps around a flexible investigation. More autonomy creates more failure paths, including choosing the wrong tool, repeating work, and stopping before verification.


## Skills, Tools and MCP

| Component | Responsibility | Build-analysis example |
|---|---|---|
| Agent | Pursue the goal and decide next steps | Investigate why a build failed |
| Skill | Provide reusable task instructions | Follow the team's diagnosis procedure |
| Tool | Execute a defined operation | Fetch a log or read a commit diff |
| MCP | Expose capabilities through a common interface | Connect the client to repository tools |

:::{div}
:class: definition-entry

Skill
: A reusable package of task-specific instructions. One skill may use several tools, and one tool may serve several skills.

:::

:::{div}
:class: definition-entry

Tool
: An executable capability with a defined input/output contract. It reads or acts on a resource; a description of an action is not the same as executing it.

:::

:::{div}
:class: definition-entry

Model Context Protocol (MCP)
: An interface through which compatible clients can discover and call exposed capabilities. The implementation behind a capability determines what it actually does.

:::

The model typically needs the tool's purpose, accepted arguments, and returned information, not every line of its implementation. Stable operations can be implemented once rather than regenerated as scripts on every request.

:::{admonition} An interface does not guarantee a correct action
:class: warning

MCP does not make a source current, a tool safe, or an agent's conclusion correct. Authentication, authorization, error handling, and result validation remain responsibilities of the application. Platform terminology and instruction-loading rules can differ.
:::


## Implementation Choices

::::{div}
:class: reading-comparison

:::{div}
### Instruction Files

Markdown makes procedures and conventions easy to review and share. Some behavior still depends on how the model interprets the instructions and how the client loads them.
:::

:::{div}
### Code Orchestration

Code can make states, transitions, retries, and checks explicit. It takes more engineering, but makes tracing and testing easier. Instruction files can still provide task guidance inside the coded process.
:::
::::

Project-scoped rules concern one workspace; broader user-scoped rules may apply across workspaces. Moving a file between platforms does not guarantee identical behavior because discovery, permissions, tools, and context assembly can differ.

| Approach | What changes? | What it does not automatically solve |
|---|---|---|
| Retrieval | Evidence supplied for this request | Faithful interpretation of that evidence |
| Fine-tuning | Model parameters through additional training | Access to the newest build log |
| Larger context | Capacity for supplied material | Noise, ranking, or source validity |
| Task decomposition | Size and scope of each step | Correct synthesis across steps |

Fine-tuning may help recurring diagnostic patterns or output behavior. The supplied paper treats its use for log analysis as a possibility requiring evaluation, not an established recommendation. New failure details still need to reach the system.


## Worked Investigation: A Failed Build

:::{admonition} Build 4182 — an illustrative investigation
:class: tip ds-example-card

**Goal:** identify the most likely change that broke the build and cite evidence. The available data includes a long log, twenty recent changes, and hundreds of changed files. These identifiers and quantities are illustrative.

1. Set a read-only diagnostic boundary: inspect the repository without modifying it.
2. Use a standard procedure to identify the failing stage and distinctive errors.
3. Fetch log sections, build metadata, diffs, and source files through tools.
4. Chunk logs by job, stage, and error boundary; preserve commit and file metadata in diffs.
5. Retrieve with both exact error codes and semantic descriptions of the failure.
6. Put the relevant log excerpt, candidate changes, and source identifiers into context.
7. Compare timing and affected components. Request another file if evidence is insufficient.
8. Report the leading hypothesis, alternatives, and uncertainty. Check that cited locations exist.

A plausible explanation is a hypothesis until the evidence or a targeted test supports it. If there are multiple independent failures, analyze them separately before synthesizing the findings.
:::
